## Data Cleaning
**Dataset:** Online Retail II (online_retail_II.xlsx)
- 1,067,371 rows across 2 sheets (Year 2009-2010 & Year 2010-2011)
- 8 columns: Invoice, StockCode, Description, Quantity, InvoiceDate, Price, Customer ID, Country

**Issues to fix:**
- 243,007 missing Customer IDs (22.8%)
- 4,382 missing Descriptions
- 34,335 duplicate rows
- 22,950 negative Quantity (returns/cancellations)
- 6,207 zero or negative Price
- 19,494 cancelled invoices (prefix 'C')
- Need TotalRevenue feature = Quantity × Price

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

Libraries loaded.


## 1. Load & merge both sheets

In [ ]:
RAW_PATH = '../data/raw/online_retail_II.xlsx'

print('Loading Year 2009-2010...')
df_2009 = pd.read_excel(RAW_PATH, sheet_name='Year 2009-2010')

print('Loading Year 2010-2011...')
df_2010 = pd.read_excel(RAW_PATH, sheet_name='Year 2010-2011')

print(f'Sheet 1: {df_2009.shape[0]:,} rows')
print(f'Sheet 2: {df_2010.shape[0]:,} rows')

df = pd.concat([df_2009, df_2010], ignore_index=True)
print(f'Merged : {df.shape[0]:,} rows  |  {df.shape[1]} columns')

Loading Year 2009-2010...
Loading Year 2010-2011...
Sheet 1: 525,461 rows
Sheet 2: 541,910 rows
Merged : 1,067,371 rows  |  8 columns


## 2. Baseline audit

In [3]:
print('=== Column types ===')
print(df.dtypes)
print()
print('=== Null counts ===')
nulls = df.isnull().sum()
pct   = (nulls / len(df) * 100).round(2)
print(pd.DataFrame({'nulls': nulls, 'pct': pct}))
print()
print(f'Duplicate rows : {df.duplicated().sum():,}')
print(f'Negative Qty   : {(df["Quantity"] < 0).sum():,}')
print(f'Zero/neg Price : {(df["Price"] <= 0).sum():,}')
print(f'Cancelled Inv  : {df["Invoice"].astype(str).str.startswith("C").sum():,}')

=== Column types ===
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID           float64
Country                object
dtype: object

=== Null counts ===
              nulls    pct
Invoice           0   0.00
StockCode         0   0.00
Description    4382   0.41
Quantity          0   0.00
InvoiceDate       0   0.00
Price             0   0.00
Customer ID  243007  22.77
Country           0   0.00

Duplicate rows : 34,335
Negative Qty   : 22,950
Zero/neg Price : 6,207
Cancelled Inv  : 19,494


## 3. Fix column names & types

In [4]:
# Standardize column names
df.columns = ['Invoice', 'StockCode', 'Description', 'Quantity',
              'InvoiceDate', 'Price', 'CustomerID', 'Country']

# Ensure correct dtypes
df['Invoice']     = df['Invoice'].astype(str).str.strip()
df['StockCode']   = df['StockCode'].astype(str).str.strip().str.upper()
df['Description'] = df['Description'].astype(str).str.strip().str.title()
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['CustomerID']  = df['CustomerID'].astype('Int64')  # nullable integer
df['Country']     = df['Country'].astype(str).str.strip()

print('Dtypes after fix:')
print(df.dtypes)

Dtypes after fix:
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
CustomerID              Int64
Country                object
dtype: object


## 4. Remove exact duplicate rows

In [5]:
before = len(df)
df = df.drop_duplicates()
after  = len(df)
print(f'Removed {before - after:,} duplicate rows  ({before:,} → {after:,})')

Removed 34,335 duplicate rows  (1,067,371 → 1,033,036)


## 5. Flag & separate cancelled invoices

In [6]:
# Flag cancellations (Invoice starts with 'C')
df['IsCancelled'] = df['Invoice'].str.startswith('C')

# Keep cancellations separate for analysis if needed
df_cancelled = df[df['IsCancelled']].copy()
df_sales     = df[~df['IsCancelled']].copy()

print(f'Cancellations : {len(df_cancelled):,} rows')
print(f'Valid sales   : {len(df_sales):,} rows')

Cancellations : 19,104 rows
Valid sales   : 1,013,932 rows


## 6. Remove negative Quantity & zero/negative Price

In [7]:
# In valid sales, Quantity and Price must be positive
before = len(df_sales)

df_sales = df_sales[df_sales['Quantity'] > 0]
df_sales = df_sales[df_sales['Price']    > 0]

after = len(df_sales)
print(f'Removed {before - after:,} rows with invalid Quantity/Price  ({before:,} → {after:,})')

Removed 6,019 rows with invalid Quantity/Price  (1,013,932 → 1,007,913)


## 7. Handle missing Customer IDs

In [8]:
missing_cust = df_sales['CustomerID'].isnull().sum()
print(f'Rows with missing CustomerID: {missing_cust:,} ({missing_cust/len(df_sales)*100:.1f}%)')

# Strategy: keep them but flag — they're still valid sales for revenue forecasting
df_sales['HasCustomerID'] = df_sales['CustomerID'].notna()

print('Kept all rows. Flagged missing CustomerID with HasCustomerID = False.')
print('These rows will be excluded from customer-level analysis but included in sales totals.')

Rows with missing CustomerID: 228,488 (22.7%)
Kept all rows. Flagged missing CustomerID with HasCustomerID = False.
These rows will be excluded from customer-level analysis but included in sales totals.


## 8. Handle missing Descriptions

In [9]:
# Fill missing descriptions using most common description per StockCode
desc_map = (
    df_sales[df_sales['Description'].notna()]
    .groupby('StockCode')['Description']
    .agg(lambda x: x.mode()[0] if len(x) > 0 else 'Unknown')
)

mask = df_sales['Description'].isna() | (df_sales['Description'] == 'Nan')
df_sales.loc[mask, 'Description'] = df_sales.loc[mask, 'StockCode'].map(desc_map)
df_sales['Description'] = df_sales['Description'].fillna('Unknown')

print(f'Remaining missing descriptions: {df_sales["Description"].isna().sum()}')

Remaining missing descriptions: 0


## 9. Remove test/noise StockCodes

In [10]:
# Non-product StockCodes (postage, manual, bank charges etc.)
noise_codes = ['POST', 'D', 'DOT', 'M', 'BANK CHARGES', 'PADS', 'AMAZONFEE', 'C2', 'CRUK']

before = len(df_sales)
df_sales = df_sales[~df_sales['StockCode'].isin(noise_codes)]
print(f'Removed {before - len(df_sales):,} noise/service rows')

# Also remove StockCodes that are purely numeric and very short (test entries)
df_sales = df_sales[df_sales['StockCode'].str.len() >= 4]
print(f'Final sales rows: {len(df_sales):,}')

Removed 4,443 noise/service rows
Final sales rows: 1,003,466


## 10. Feature engineering

In [11]:
# Core revenue metric
df_sales['TotalRevenue'] = (df_sales['Quantity'] * df_sales['Price']).round(2)

# Date parts for time series & SQL dimensions
df_sales['Year']        = df_sales['InvoiceDate'].dt.year
df_sales['Month']       = df_sales['InvoiceDate'].dt.month
df_sales['YearMonth']   = df_sales['InvoiceDate'].dt.to_period('M').astype(str)
df_sales['DayOfWeek']   = df_sales['InvoiceDate'].dt.dayofweek   # 0=Mon
df_sales['DayOfWeekName'] = df_sales['InvoiceDate'].dt.day_name()
df_sales['Hour']        = df_sales['InvoiceDate'].dt.hour
df_sales['Quarter']     = df_sales['InvoiceDate'].dt.quarter
df_sales['WeekOfYear']  = df_sales['InvoiceDate'].dt.isocalendar().week.astype(int)

print('New columns added:')
print([c for c in df_sales.columns if c not in ['Invoice','StockCode','Description',
       'Quantity','InvoiceDate','Price','CustomerID','Country','IsCancelled']])

New columns added:
['HasCustomerID', 'TotalRevenue', 'Year', 'Month', 'YearMonth', 'DayOfWeek', 'DayOfWeekName', 'Hour', 'Quarter', 'WeekOfYear']


## 11. Final quality check

In [12]:
print('=== FINAL DATASET ===')
print(f'Shape      : {df_sales.shape}')
print(f'Date range : {df_sales["InvoiceDate"].min()} → {df_sales["InvoiceDate"].max()}')
print(f'Countries  : {df_sales["Country"].nunique()}')
print(f'Customers  : {df_sales["CustomerID"].nunique()}')
print(f'Products   : {df_sales["StockCode"].nunique()}')
print(f'Invoices   : {df_sales["Invoice"].nunique()}')
print(f'Total Rev  : £{df_sales["TotalRevenue"].sum():,.2f}')
print()
print('Remaining nulls:')
print(df_sales.isnull().sum()[df_sales.isnull().sum() > 0])
print()
df_sales.head()

=== FINAL DATASET ===
Shape      : (1003466, 19)
Date range : 2009-12-01 07:45:00 → 2011-12-09 12:50:00
Countries  : 43
Customers  : 5861
Products   : 4735
Invoices   : 39565
Total Rev  : £19,655,472.77

Remaining nulls:
CustomerID    226842
dtype: int64



,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,IsCancelled,HasCustomerID,TotalRevenue,Year,Month,YearMonth,DayOfWeek,DayOfWeekName,Hour,Quarter,WeekOfYear
0,489434,85048,15Cm Christmas Glass Ball 20 Lights,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,False,True,83.4,2009,12,2009-12,1,Tuesday,7,4,49
1,489434,79323P,Pink Cherry Lights,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False,True,81.0,2009,12,2009-12,1,Tuesday,7,4,49
2,489434,79323W,White Cherry Lights,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False,True,81.0,2009,12,2009-12,1,Tuesday,7,4,49
3,489434,22041,"Record Frame 7"" Single Size",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,False,True,100.8,2009,12,2009-12,1,Tuesday,7,4,49
4,489434,21232,Strawberry Ceramic Trinket Box,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,False,True,30.0,2009,12,2009-12,1,Tuesday,7,4,49


## 12. Save cleaned data

In [13]:
os.makedirs('../data/processed', exist_ok=True)

# Main cleaned file for EDA & modelling
df_sales.to_csv('../data/processed/cleaned_sales.csv', index=False)

# Monthly aggregated — ready for forecasting
monthly = (
    df_sales.groupby('YearMonth')
    .agg(
        TotalRevenue   = ('TotalRevenue', 'sum'),
        TotalQuantity  = ('Quantity', 'sum'),
        NumInvoices    = ('Invoice', 'nunique'),
        NumCustomers   = ('CustomerID', 'nunique'),
        NumProducts    = ('StockCode', 'nunique')
    )
    .reset_index()
    .sort_values('YearMonth')
)
monthly.to_csv('../data/processed/monthly_sales.csv', index=False)

# Monthly by category (StockCode level for category forecasting)
monthly_cat = (
    df_sales.groupby(['YearMonth', 'StockCode', 'Description'])
    .agg(Revenue=('TotalRevenue','sum'), Qty=('Quantity','sum'))
    .reset_index()
    .sort_values(['StockCode', 'YearMonth'])
)
monthly_cat.to_csv('../data/processed/monthly_by_product.csv', index=False)

print('Saved:')
print('  → data/processed/cleaned_sales.csv')
print('  → data/processed/monthly_sales.csv')
print('  → data/processed/monthly_by_product.csv')
print(f'Monthly rows: {len(monthly)}')
print(monthly)

Saved:
  → data/processed/cleaned_sales.csv
  → data/processed/monthly_sales.csv
  → data/processed/monthly_by_product.csv
Monthly rows: 25
   YearMonth  TotalRevenue  TotalQuantity  NumInvoices  NumCustomers  \
0    2009-12     798232.03         425252         1671           952   
1    2010-01     621353.43         390471         1089           718   
2    2010-02     537926.69         381621         1189           771   
3    2010-03     761748.53         525045         1647          1051   
4    2010-04     646474.06         366075         1436           939   
5    2010-05     643585.64         395442         1484           965   
6    2010-06     697382.32         406243         1615          1035   
7    2010-07     633079.42         337560         1507           924   
8    2010-08     674192.89         471939         1402           910   
9    2010-09     869277.16         583368         1789          1136   
10   2010-10    1094563.63         619515         2244          1492